In [3]:
!pip install rouge-score bert-score torch

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached torch-2.12.1-cp314-cp314-win_amd64.whl.metadata (31 kB)
  Using cached transformers-5.12.1-py3-none-any.whl.metadata (33 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached filelock-3.29.4-py3-none-any.whl.metadata (2.0 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.6.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cach

In [1]:
!pip install ollama psutil nltk

   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.6 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.6 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.6 MB 599.9 kB/s eta 0:00:02
   ------------- -------------------------- 0.5/1.6 MB 599.9 kB/s eta 0:00:02
   -------------------- ------------------- 0.8/1.6 MB 657.8 kB/s eta 0:00:02
   -------------------- ------------------- 0.8/1.6 MB 657.8 kB/s eta 0:00:02
   --------------------------- ------------ 1.0/1.6 MB 699.0 kB/s eta 0:00:01
   --------------------------------- ------ 1.3/1.6 MB 706.6 kB/s eta 0:00:01
   --------------------------------- ------ 1.3/1.6 MB 706.6 kB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 711.9 kB/s  0:00:02
   -------------------------------

In [1]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [4]:
from rouge_score import rouge_scorer
from bert_score import score as bertscore

c:\Users\User\anaconda3\envs\condaenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
from rouge_score import rouge_scorer
from bert_score import score as bertscore_calc

def evaluate_summary(generated_summary, reference_summary):
    # Guard rail against empty generations or errors
    if not generated_summary or not reference_summary or "Error during inference:" in str(generated_summary):
        return {
            "ROUGE1": 0.0,
            "ROUGE2": 0.0,
            "ROUGEL": 0.0,
            "BERTScore": 0.0
        }

    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(str(reference_summary), str(generated_summary))

    try:
        # Utilizing a fast lightweight backbone to calculate semantic embeddings quickly
        P, R, F1 = bertscore_calc(
            [str(generated_summary)], 
            [str(reference_summary)], 
            lang="en", 
            model_type="microsoft/deberta-v3-small", 
            verbose=False
        )
        bert_f1 = F1.mean().item()
    except Exception:
        bert_f1 = 0.0

    return {
        "ROUGE1": scores['rouge1'].fmeasure,
        "ROUGE2": scores['rouge2'].fmeasure,
        "ROUGEL": scores['rougeL'].fmeasure,
        "BERTScore": bert_f1
    }

In [8]:
import time
import os
import psutil
import pandas as pd
import ollama
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Ensure necessary NLP resources are downloaded
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

# 1. LOAD YOUR KAGGLE DATASET
csv_path = 'datasets/scisumm.csv' 

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Please make sure the dataset file is named '{csv_path}' and placed in the correct path.")

df = pd.read_csv(csv_path)
print("Dataset columns:", df.columns.tolist())

# Dynamically map text and summary columns based on standard naming variations
text_column = 'text' if 'text' in df.columns else ('document' if 'document' in df.columns else df.columns[0])

if 'summary' in df.columns:
    summary_column = 'summary'
elif 'abstract' in df.columns:
    summary_column = 'abstract'
else:
    summary_column = df.columns[1] # Fallback to the second column

print(f"Mapping Input Text to: '{text_column}' | Mapping Ground Truth Summary to: '{summary_column}'")

# Process the first 10 papers for testing
sample_df = df.head(10)

# 2. PROMPT COMPRESSION FUNCTION
def compress_prompt(text):
    if not isinstance(text, str):
        return ""
    stop_words = set(stopwords.words('english'))
    word_tokens = word_tokenize(text)
    compressed_tokens = [w for w in word_tokens if not w.lower() in stop_words]
    return " ".join(compressed_tokens)

# 3. EXPERIMENTAL TESTING ENGINE
results = []

def run_test(pipeline_name, model_name, input_text, reference_summary, paper_id):
    truncated_text = " ".join(str(input_text).split()[:800])
    
    process = psutil.Process()
    start_mem = process.memory_info().rss / (1024 * 1024) # MB
    start_time = time.time()
    
    try:
        response = ollama.chat(model=model_name, messages=[
            {'role': 'user', 'content': f"Summarize this scientific text in two sentences: {truncated_text}"}
        ])
        output_text = response['message']['content']
    except Exception as e:
        output_text = f"Error during inference: {str(e)}"
        
    end_time = time.time()
    end_mem = process.memory_info().rss / (1024 * 1024) # MB
    
    latency = end_time - start_time
    memory_used = max(0, end_mem - start_mem)

    metrics = evaluate_summary(output_text, reference_summary)
    
    return {
        'Paper_ID': paper_id,
        'Pipeline': pipeline_name,
        'Latency_Sec': round(latency, 3),
        'RAM_Used_MB': round(memory_used, 2),
        'Output_Word_Count': len(output_text.split()),
        "Compression_Ratio": round(len(output_text.split()) / max(1, len(truncated_text.split())), 4),
        "ROUGE1": round(metrics["ROUGE1"], 4),
        "ROUGE2": round(metrics["ROUGE2"], 4),
        "ROUGEL": round(metrics["ROUGEL"], 4),
        "BERTScore": round(metrics["BERTScore"], 4)
    }

# 4. RUN THE COMPARATIVE EXPERIMENT LOOP
print("\nStarting optimization benchmarks...")

for idx, row in sample_df.iterrows():
    raw_text = row[text_column]
    ref_summary = row[summary_column]
    print(f"Processing Paper {idx + 1}/10...", end="\r")
    
    # Baseline
    res_base = run_test("Baseline", "phi3", raw_text, ref_summary, idx)
    
    # Pipeline A: Quantized Model
    res_pipe_a = run_test("Quantized_Model", "phi3:3.8b-mini-4k-instruct-q4_K_M", raw_text, ref_summary, idx)
    
    # Pipeline B: Prompt Compression
    compressed_text = compress_prompt(raw_text)
    res_pipe_b = run_test("Prompt_Compression", "phi3", compressed_text, ref_summary, idx)

    # Pipeline C: Quantized Model + Prompt Compression (Hybrid)
    res_pipe_c = run_test("Quantized_Prompt_Compression", "phi3:3.8b-mini-4k-instruct-q4_K_M", compressed_text, ref_summary, idx)
    
    results.extend([res_base, res_pipe_a, res_pipe_b, res_pipe_c])

print("\nAll experiments complete successfully!")

Dataset columns: ['text', 'summary']
Mapping Input Text to: 'text' | Mapping Ground Truth Summary to: 'summary'

Starting optimization benchmarks...


c:\Users\User\anaconda3\envs\condaenv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--microsoft--deberta-v3-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
[transformers] Could not extract SentencePiece model from C:\Users\User\.cache\huggingface\hub\models--mi

KeyboardInterrupt: 

In [9]:
# 5. CONVERT THE LOGGED METRICS INTO A DATAFRAME
results_df = pd.DataFrame(results)

# Pivot all dependent numeric variables under test
pivot_df = results_df.pivot(
    index='Paper_ID',
    columns='Pipeline',
    values=['Latency_Sec', 'RAM_Used_MB', 'Output_Word_Count', 'ROUGE1', 'ROUGE2', 'ROUGEL', 'BERTScore']
)

# Swap structural levels so Pipeline is on top
pivot_df = pivot_df.swaplevel(0, 1, axis=1)

pipeline_order = [
    "Baseline",
    "Quantized_Model",
    "Prompt_Compression",
    "Quantized_Prompt_Compression"
]

metric_order = [
    "Latency_Sec",
    "RAM_Used_MB",
    "Output_Word_Count",
    "ROUGE1",
    "ROUGE2",
    "ROUGEL",
    "BERTScore"
]

# Reindex layout structure cleanly
pivot_df = pivot_df.reindex(
    columns=pd.MultiIndex.from_product([pipeline_order, metric_order])
)

# Calculate averages for performance columns
average_row = pivot_df.mean()
pivot_df.loc["Average"] = average_row

# Export the clean matrix straight to file
pivot_df.to_csv("llm_optimization_results.csv")

# Display results
pivot_df.head()

Baseline                                                        \
         Latency_Sec RAM_Used_MB Output_Word_Count  ROUGE1  ROUGE2  ROUGEL   
Paper_ID                                                                     
0             18.667         0.0              75.0  0.3819  0.0812  0.2312   
Average       18.667         0.0              75.0  0.3819  0.0812  0.2312   

                   Quantized_Model                                ...  \
         BERTScore     Latency_Sec RAM_Used_MB Output_Word_Count  ...   
Paper_ID                                                          ...   
0              0.0          33.549         0.0              99.0  ...   
Average        0.0          33.549         0.0              99.0  ...   

         Prompt_Compression                   Quantized_Prompt_Compression  \
                     ROUGE2  ROUGEL BERTScore                  Latency_Sec   
Paper_ID                                                                     
0                     0.129  0.2553       0.0                       15.879   
Average               0.129  0.2553       0.0                       15.879   

                                                                          
         RAM_Used_MB Output_Word_Count  ROUGE1  ROUGE2  ROUGEL BERTScore  
Paper_ID                                                                  
0                0.0              68.0  0.4767  0.1361  0.2487       0.0  
Average          0.0              68.0  0.4767  0.1361  0.2487       0.0  

[2 rows x 28 columns]

In [ ]:
# 4th Cell: Extract and clean generated text outputs
summary_df = results_df.pivot(
    index="Paper_ID",
    columns="Pipeline",
    values="Summary"
)

# Optional clean-up: Replace any error messages with a blank string or "Inference Error" 
# so your final appendix table looks highly professional
for col in summary_df.columns:
    summary_df[col] = summary_df[col].apply(lambda x: "Inference Error" if "Error during inference:" in str(x) else x)

# Display a preview of the text summaries inside your notebook
print("Generated Summaries Preview:")
print(summary_df.head())

# Save the text summaries matrix to a separate CSV
summary_df.to_csv("llm_generated_summaries.csv")
print("\nText output matrix saved cleanly to 'llm_generated_summaries.csv'.")

KeyError: 'Summary'